In [1]:
import os
import pandas as pd
from tqdm import tqdm


In [ ]:
folder = "/home/student/rishi/joined"
full_index = pd.date_range('2023-01-01 00:00:00', '2025-12-31 23:00:00', freq='h')
features = ["PM2.5 (µg/m³)", "PM10 (µg/m³)", "NO2 (µg/m³)", "SO2 (µg/m³)", "CO (mg/m³)", "Ozone (µg/m³)"]
pollutant_dicts_dir = "visualize_2023_india_dicts"
os.makedirs(pollutant_dicts_dir, exist_ok=True)
image_dir = "visualize_2023_india_images"
os.makedirs(image_dir, exist_ok=True)

In [ ]:
def generate_comparison_df(feature, folder, full_index):
    feature_key = feature.split(" ")[0]
    site_dict = {}
    files = [f for f in sorted(os.listdir(folder))
             if os.path.isfile(os.path.join(folder, f)) and feature_key in f]
    for file in files:
        df = pd.read_csv(os.path.join(folder, file))
        df['Timestamp'] = pd.to_datetime(df['Timestamp'])
        df = df.set_index('Timestamp').reindex(full_index)
        site_dict[file] = df[feature]
    return pd.DataFrame(site_dict)


In [ ]:
for feature in tqdm(features):
    safe = feature.replace("/", "_").replace(" ", "_")
    feature_df = generate_comparison_df(feature, folder, full_index)
    feature_df.to_csv(f"{pollutant_dicts_dir}/{safe}_df.csv")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

limit_dict = {"PM2.5 (µg/m³)": 60, "PM10 (µg/m³)": 100, "NO2 (µg/m³)": 80, "SO2 (µg/m³)": 80, "CO (mg/m³)": 4, "Ozone (µg/m³)": 180, "NH3 (µg/m³)": 400}


def plot_site_comparison_heatmap(df, feature_name):
    nan_counts = df.isnull().sum().sort_values()
    df = df.reindex(columns=nan_counts.index)
    df_transposed = df.T

    fig, ax = plt.subplots(figsize=(20, len(df.columns) * 0.025))
    vmax = limit_dict[feature_name] if feature_name in limit_dict else df.max().max()
    sns.heatmap(
        df_transposed,
        cmap='YlOrRd',
        cbar_kws={'label': f'{feature_name}'},
        xticklabels=False,
        yticklabels=False,
        ax=ax,
        mask=df_transposed.isna(),
        vmin=0,
        vmax=vmax
    )

    years = pd.to_datetime(df.index).year.unique()
    year_starts = [df.index.get_loc(pd.Timestamp(f'{year}-01-01'))
                   for year in years if pd.Timestamp(f'{year}-01-01') in df.index]
    ax.set_xticks(year_starts)
    ax.set_xticklabels(years, rotation=0, fontsize=10)

    plt.title(f'{feature_name} Across Sites Over Time', fontsize=16, pad=20)
    plt.xlabel('Year', fontsize=12)
    plt.ylabel('Site', fontsize=12)
    plt.tight_layout()

    safe = feature_name.split(" ")[0]
    plt.savefig(f"{image_dir}/{safe}.png", dpi=500, bbox_inches='tight')
    plt.show()

In [ ]:
for feature in tqdm(features):
    safe = feature.replace("/", "_").replace(" ", "_")
    feature_df = pd.read_csv(f"{pollutant_dicts_dir}/{safe}_df.csv", index_col=0, parse_dates=True)
    plot_site_comparison_heatmap(feature_df, feature_name=feature)